# 1. What an embedding learns

**Goal:** connect linear algebra and differentiation to the trainable part of Pose Embed. Check cosine retrieval, derive the gradient through normalization, and take one transparent learning step.

**All numbers are invented.** This notebook uses a 3-to-2 linear map and a simple target-alignment loss. The study uses an 8,704-to-2,048 biased linear head, the three specified recipes, and AdamW. This example is neither a reduced scientific run nor an implementation of contextual loss. NumPy is the only dependency; cells do not read or write files, contact the cluster, or download models.

Run from top to bottom. Before each cell, predict the result in words.

## From vectors to a retrieval decision

An **embedding** is a vector whose geometry is useful for comparison. Its coordinates need not each have a human-readable meaning. A gallery holds one reference per action; a query is compared against all references.

For nonzero vectors,
\[
s(q,g)=\frac{q^Tg}{\|q\|_2\|g\|_2}.
\]
After unit normalization this is a dot product, and
\[
\|\hat q-\hat g\|_2^2=2-2\hat q^T\hat g.
\]
Thus maximizing cosine and minimizing squared Euclidean distance give the same ranking **for unit vectors**. Multiplying a query by a positive scalar cannot change that ranking.

In [1]:
import numpy as np

def unit(vectors):
    norms = np.linalg.norm(vectors, axis=-1, keepdims=True)
    assert np.all(norms > 0), 'These teaching examples must be nonzero.'
    return vectors / norms

query = np.array([3.0, 4.0])
gallery = np.array([[1.0, 0.0], [0.0, 1.0], [-1.0, 0.0]])
labels = np.array(['reference A', 'reference B', 'reference C'])
q, g = unit(query), unit(gallery)
cosine = g @ q
squared_distance = np.sum((g - q) ** 2, axis=1)
cosine_order = np.argsort(-cosine, kind='stable')
distance_order = np.argsort(squared_distance, kind='stable')
assert np.allclose(squared_distance, 2 - 2 * cosine)
assert np.array_equal(cosine_order, distance_order)
assert np.allclose(unit(10 * query), q)
print('Cosines:', cosine)
print('Squared distances:', squared_distance)
print('Top match:', labels[cosine_order[0]])

Cosines: [ 0.6  0.8 -0.6]
Squared distances: [0.8 0.4 3.2]
Top match: reference B


## What is trainable?

Let $h$ be a cached feature vector from the frozen encoder. The head computes
\[
a=Wh+b,\qquad z=\frac{a}{\|a\|_2}.
\]
The encoder's weights and $h$ stay fixed during head training. $W$ and $b$ change. With a bias, this is an **affine** map before normalization; it is conventionally called a linear layer.

For a simple teaching target $t$ of unit length, choose $L(W,b)=1-z^Tt$. Lower loss means a larger cosine with the target. The real recipes instead use labels to define relationships among examples.

Away from $a=0$, the derivative of normalization is
\[
\frac{\partial z}{\partial a}=\frac{I-zz^T}{\|a\|_2}.
\]
Applying the chain rule gives
\[
\nabla_aL=-\frac{t-z(z^Tt)}{\|a\|_2},\qquad
\nabla_WL=(\nabla_aL)h^T,\qquad \nabla_bL=\nabla_aL.
\]
The component parallel to $z$ is removed. At first order, changing only the length of $a$ cannot change its unit direction.

In [2]:
h = np.array([1.0, 2.0, -1.0])
W = np.array([[0.4, 0.2, -0.1], [-0.2, 0.1, 0.3]])
b = np.array([0.1, 0.2])
target = np.array([0.0, 1.0])

def loss(weights, bias):
    return float(1.0 - unit(weights @ h + bias) @ target)

a = W @ h + b
z = unit(a)
grad_a = -(target - z * (z @ target)) / np.linalg.norm(a)
grad_W = np.outer(grad_a, h)
grad_b = grad_a
assert np.isclose(grad_a @ a, 0.0)
assert grad_W.shape == W.shape and grad_b.shape == b.shape
print('Feature / weight / embedding shapes:', h.shape, W.shape, z.shape)
print(f'Initial cosine with target: {z @ target:.6f}')
print(f'Initial loss: {loss(W, b):.6f}')

Feature / weight / embedding shapes: (3,) (2, 3) (2,)
Initial cosine with target: -0.099504
Initial loss: 1.099504


## Check the derivative independently

For parameter $\theta_j$, a central finite difference estimates
\[
\frac{\partial L}{\partial\theta_j}\approx
\frac{L(\theta+\varepsilon e_j)-L(\theta-\varepsilon e_j)}{2\varepsilon}.
\]
The approximation is accurate here because the loss is smooth near these values and $\varepsilon$ is modest. This does **not** verify the real study's nonsmooth contextual backward rules, optimizer, or GPU arithmetic.

In [3]:
epsilon = 1e-6
numerical_W = np.zeros_like(W)
numerical_b = np.zeros_like(b)
for index in np.ndindex(W.shape):
    direction = np.zeros_like(W)
    direction[index] = epsilon
    numerical_W[index] = (loss(W + direction, b) - loss(W - direction, b)) / (2 * epsilon)
for index in range(b.size):
    direction = np.zeros_like(b)
    direction[index] = epsilon
    numerical_b[index] = (loss(W, b + direction) - loss(W, b - direction)) / (2 * epsilon)
error = max(np.max(np.abs(grad_W - numerical_W)), np.max(np.abs(grad_b - numerical_b)))
assert np.allclose(grad_W, numerical_W, rtol=1e-6, atol=1e-8)
assert np.allclose(grad_b, numerical_b, rtol=1e-6, atol=1e-8)
print(f'Maximum analytic / numerical gradient difference: {error:.3e}')

Maximum analytic / numerical gradient difference: 7.147e-11


## Take one learning step

Gradient descent updates parameters in the direction that locally lowers the loss:
\[
W_{\rm new}=W-\eta\nabla_WL,\qquad b_{\rm new}=b-\eta\nabla_bL.
\]
The learning rate $\eta$ controls the step size. This is one plain gradient step so every operation is visible. AdamW, used by the project, also keeps running gradient statistics and applies weight decay. A lower training loss on one invented example is not evidence of generalization.

In [4]:
learning_rate = 0.05
new_W = W - learning_rate * grad_W
new_b = b - learning_rate * grad_b
before, after = loss(W, b), loss(new_W, new_b)
assert after < before
assert np.array_equal(h, np.array([1.0, 2.0, -1.0]))
print(f'Loss: {before:.6f} -> {after:.6f}')
print(f'Cosine: {1 - before:.6f} -> {1 - after:.6f}')
print('Only W and b changed; the encoder feature h stayed fixed.')

Loss: 1.099504 -> 0.769706
Cosine: -0.099504 -> 0.230294
Only W and b changed; the encoder feature h stayed fixed.


## Explain it without the code

1. **Why normalize?** To compare direction rather than magnitude and make cosine retrieval a dot product.
2. **Why can learning change a cosine when scaling cannot?** A general matrix and bias change directions and relative geometry; they do more than apply one scalar.
3. **Does a frozen encoder mean nothing learns?** No. The head's weights and bias learn; the encoder does not.
4. **What did the gradient check establish?** Agreement between an analytic derivative and finite differences for this smooth toy loss at these inputs.
5. **What does it not establish?** Correctness of the production recipes, stability over epochs, or better retrieval on unseen actions.

**Connection to the study:** [final v3 plan, representation and training](../../plan/research-plan.v3.md). The authoritative implementation remains the project source; this notebook isolates one prerequisite idea.